# Modelo Base y Modelo Predictivo

Este notebook parte de `train.csv` y `test.csv` que contienen las variables numéricas estandarizadas y las variables categóricas codificadas.

## Imports
---

In [6]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

RANDOM_STATE = 42


### Explicación de los Imports

Antes de trabajar con los datos se importan las librerías necesarias:

- **numpy:** librería base de Python para hacer cálculos numéricos rápidos. En este notebook la mayoría de las demás librerías la usa por debajo.

- **pandas:** permite trabajar con los datos en forma de tablas. En este notebook, se usa para cargar los archivos `train.csv` y `test.csv`, y organizar resultados.

- **DummyClassifier:** modelo que se usa como punto de comparación para el modelo base. No aprende nada de los datos, solo aplica reglas simples.

- **Métricas (`accuracy_score`, `precision_score`, `recall_score`, `f1_score`, `roc_auc_score`):** funciones que calculan qué tan bien está funcionando un modelo, cada una midiendo un aspecto distinto del desempeño.

### Justificación de RANDOM_STATE
Muchos de los procesos que vamos a usar tienen un componente de aleatoriedad interno, lo que significa que, si se ejecuta el mismo código dos veces, se podrían obtener resultados ligeramente distintos cada vez, lo cual va en contra de la reproducibilidad.

La solución a este problema es usar una semilla aleatoria con un número fijo, lo que garantizará que siempre que se use la misma semilla, el resultado de un proceso sea siempre idéntico entre una ejecución y otra.

## Carga de los Datos ya Procesados
---

En esta celda se cargan los archivos `train.csv` y `test.csv` obtenidos anteriormente, y se separan en dos partes cada uno:

- **`X`:** tabla con todas las variables predictoras.

- **`y`:** columna `Churn`, es decir, la respuesta correcta que el modelo debe aprender a predecir.

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Separar las variables predictoras y la variable objetivo
X_train = train.drop(columns=["Churn"])
y_train = train["Churn"]
X_test = test.drop(columns=["Churn"])
y_test = test["Churn"]

print("Tamaño de entrenamiento (X_train):", X_train.shape, " | Tamaño de prueba (X_test):", X_test.shape)
print("Proporción de clientes que se van (Churn) en entrenamiento:")
print(y_train.value_counts(normalize=True).round(3))

Tamaño de entrenamiento (X_train): (5634, 45)  | Tamaño de prueba (X_test): (1409, 45)
Proporción de clientes que se van (Churn) en entrenamiento:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


### Interpretación de los Resultados
El resultado obtenido muestra que el conjunto de entrenamiento tiene 5634 clientes y 45 variables predictoras, y, que el de prueba tiene 1409 clientes con esas mismas variables. La proporción de la clase `Churn` (0 = se queda, 1 = abandona el servicio) es de aproximadamente 73.5% / 26.5%, es decir, el dataset está desbalanceado: hay casi 3 clientes que se quedan por cada uno que se va.

Este desbalance no es un problema que haya que arreglar obligatoriamente, pero sí es algo que hay que tener muy presente durante todo este notebook, porque cambia por completo qué significa que un modelo sea "bueno". Por ejemplo: un modelo que nunca identificara a ningún cliente que se va tendría igual un 73.5% de aciertos generales (*accuracy*), y eso sonaría bien en un primer vistazo, pero sería completamente inútil para el objetivo real del proyecto que es anticipar la fuga de clientes.



## Modelo Base (baseline)
---

Antes de construir un modelo predictivo, se construye uno extremadamente simple que no aprende nada de los patrones de los datos. Su función no es predecir bien, sino servir como punto de comparación mínimo. La pregunta que responde esta sección es: **"si el modelo predictivo que construyamos no logra superar claramente a esta regla tan simple, ¿realmente vale la pena usar Machine Learning para este problema?"**

`scikit-learn` ofrece un `DummyClassifier` con tres estrategias distintas:

1. **`most_frequent`:** siempre predice la clase que más se repite en los datos de entrenamiento. En este caso, siempre predeciría `Churn = No`.

2. **`stratified`:** genera predicciones al azar, pero respetando las proporciones reales de cada clase, es decir, predice "abandona el servicio" aproximadamente el 26.5% de las veces, al azar.

3. **`uniform`:** genera predicciones completamente al azar, con 50% de probabilidad para cada clase, sin tener en cuenta cuál es más común.

A continuación se calculan y comparan las tres, para elegir cuál es el baseline más adecuado.


In [11]:
estrategias = ["most_frequent", "stratified", "uniform"]
resumen_estrategias = []

# Se entrena y evalúa un DummyClassifier para cada estrategia
for estrategia in estrategias:
  dummy = DummyClassifier(strategy=estrategia, random_state=RANDOM_STATE)
  dummy.fit(X_train, y_train)
  y_pred_d = dummy.predict(X_test)
  y_proba_d = dummy.predict_proba(X_test)[:, 1]
  
  resumen_estrategias.append({
    "estrategia": estrategia,
    "accuracy": accuracy_score(y_test, y_pred_d),
		"precision": precision_score(y_test, y_pred_d, zero_division=0),
		"recall": recall_score(y_test, y_pred_d, zero_division=0),
		"f1": f1_score(y_test, y_pred_d, zero_division=0),
		"roc_auc": roc_auc_score(y_test, y_proba_d),
  })

pd.DataFrame(resumen_estrategias).set_index("estrategia").round(3)

,accuracy,precision,recall,f1,roc_auc
estrategia,,,,,
most_frequent,0.735,0.000,0.000,0.00,0.500
stratified,0.622,0.289,0.291,0.29,0.516
uniform,0.487,0.252,0.476,0.33,0.500


### Interpretación de la Tabla
Cada fila es una de las tres estrategias y cada columna es una métrica distinta:

- **accuracy:** porcentaje total de aciertos.

- **precision:** de los clientes que el modelo dijo que se iban, cuántos realmente se fueron.

- **recall:** de los clientes que realmente se fueron, cuántos detectó el modelo.

- **f1:** combina precision y recall en un solo número.

- **roc_auc:** mide qué tan bien distingue el modelo entre ambas clases en general, donde 0.5 equivale a adivinar al azar.

### Interpretación de los Resultados
`most_frequent` obtiene el mejor *accuracy* (~ 73%), pero su *precision*, *recall* y *f1* son exactamente 0, porque, al predecir siempre "No" para todos los clientes, nunca acierta ni una sola vez con los que sí se van.

Las otras dos estrategias (`stratified` y `uniform`) sí llegan a acertar algunos casos de churn por pura casualidad, pero a costa de un *accuracy* mucho más bajo y sin una capacidad predictiva real, ya que su *roc_auc* ronda 0.5, el valor esperado de una predicción aleatoria.

### Elección del Baseline
Se elige `most_frequent` como baseline oficial, ya que, aunque suene contradictorio elegir la estrategia que "más falla" en detectar clientes en riesgo, es justamente la comparación más exigente y realista: el costo de no tener ningún modelo en la empresa hoy, es decir, asumir que nadie se va y no hacer nada al respecto.